# KAVALAN — Weapon Detection Fine-tune (YOLOv8)

**Run this as a Kaggle Notebook** — see `ml/README.md` in the repo for setup steps.

1. Add Input dataset: `iqmansingh/guns-knives-object-detection`
2. Settings → Accelerator → GPU T4 x2
3. Run All

Goal: fine-tune a pretrained YOLOv8 (COCO weights) to detect guns/knives in evidence
photos — a new capability that turns image evidence into structured detections
(class + bounding box + confidence) instead of the free-text extraction the Groq vision
call currently produces in `src/app/api/cases/[id]/digital/extract-image/route.ts`.

In [ ]:
!pip install -q ultralytics

## Step 1 — Inspect what Kaggle actually mounted

**Run this first.** YOLO datasets on Kaggle are usually already in YOLO format
(`train/images`, `train/labels`, `valid/images`, `valid/labels`, plus a `data.yaml`), but
the exact folder names/nesting vary by uploader — read the printed tree before Step 2.

In [ ]:
import os

INPUT_ROOT = "/kaggle/input"
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root.replace(INPUT_ROOT, "").count(os.sep)
    if depth > 4:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    if depth == 4:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

print()
print("Any existing data.yaml files:")
import glob

for f in glob.glob(f"{INPUT_ROOT}/**/*.yaml", recursive=True) + glob.glob(
    f"{INPUT_ROOT}/**/*.yml", recursive=True
):
    print(" ", f)

## Step 2 — Point at (or build) `data.yaml`

If Step 1 found an existing `data.yaml`, set `EXISTING_YAML` to that path and skip to
Step 3. Otherwise this cell writes one, assuming the standard
`train/images,labels` + `valid/images,labels` layout — **adjust `DATASET_DIR` and the
class names to match what Step 1 printed.**

In [ ]:
# TODO: adjust to match Step 1's output.
DATASET_DIR = "/kaggle/input/guns-knives-object-detection"
EXISTING_YAML = None  # set to a path string if Step 1 found one, else leave None

if EXISTING_YAML:
    DATA_YAML = EXISTING_YAML
else:
    DATA_YAML = "/kaggle/working/data.yaml"
    yaml_content = f"""
path: {DATASET_DIR}
train: train/images
val: valid/images
test: test/images

names:
  0: gun
  1: knife
"""
    with open(DATA_YAML, "w") as f:
        f.write(yaml_content)
    print(yaml_content)

print("Using:", DATA_YAML)

## Step 3 — Fine-tune

`EPOCHS` is deliberately small for a first pipeline-sanity run — bump it up (e.g.
50–100) once this completes cleanly and you've checked the val metrics below.

In [ ]:
from ultralytics import YOLO

EPOCHS = 25
IMG_SIZE = 640

model = YOLO("yolov8n.pt")  # nano — fastest to fine-tune, good for a first pass

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=16,
    project="/kaggle/working/runs",
    name="weapon-detect",
    patience=10,
)

## Step 4 — Validate

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## Step 5 — Sanity-check a few predictions

In [ ]:
import glob
import random

val_images = glob.glob(f"{DATASET_DIR}/valid/images/*") or glob.glob(
    f"{DATASET_DIR}/test/images/*"
)
sample = random.sample(val_images, min(5, len(val_images)))
preds = model.predict(sample, save=True, project="/kaggle/working/runs", name="predict-sample")
for p in preds:
    print(p.path, "->", [model.names[int(c)] for c in p.boxes.cls])

## Step 6 — Export to ONNX

In [ ]:
best_weights = "/kaggle/working/runs/weapon-detect/weights/best.pt"
best_model = YOLO(best_weights)
onnx_path = best_model.export(format="onnx", imgsz=IMG_SIZE, opset=12)
print("Exported to:", onnx_path)
print("Download it from the Output tab (under runs/weapon-detect/weights/).")